# Add fire frequency to GEDI fire pairs

This notebook samples the MapBiomas fire-frequency raster around each pre-fire and post-fire footprint in the fire-only pair dataset created in notebook 2.

The workflow is:
1. Load the fire-only GEDI pair GeoPackage and the fire-frequency raster.
2. Rebuild pre and post footprint geometries from stored WKT.
3. Buffer each footprint in meters using the pair-specific projected CRS.
4. Mask the raster with each buffered geometry.
5. Store the maximum fire-frequency value found in each buffer.
6. Save the updated pair table.


In [ ]:
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from rasterio.mask import mask
from shapely import wkt
from shapely.geometry import mapping
from tqdm import tqdm


In [ ]:
pairs_path = "GEDI_Pairs/GEDI_Footprint_Fire_Pairs.gpkg"
ff_path = "Fire History/Mapbioma_Collection_4_Fire_Frequency_AoD_1985_2024.tif"
output_path = "GEDI_Pairs/GEDI_Footprint_Pairs_Fire_FF.gpkg"
buffer_size = 20

pre_col_name = f"pre_buff_{buffer_size}"
post_col_name = f"post_buff_{buffer_size}"


In [ ]:
filter_table = pd.DataFrame(
    [
        {"stage": "Input filter", "filter": "pair source", "value": pairs_path},
        {"stage": "Input filter", "filter": "raster source", "value": ff_path},
        {"stage": "Geometry setup", "filter": "pre/post geometry parsing", "value": "load geometry_1 and geometry_2 WKT as EPSG:4326 polygons"},
        {"stage": "Buffer filter", "filter": "buffer size", "value": f"{buffer_size} m"},
        {"stage": "Buffer filter", "filter": "buffer CRS", "value": "buffer in each row group's pairing_crs, then return to EPSG:4326"},
        {"stage": "Buffer filter", "filter": "missing pairing_crs", "value": "skip rows in groups with missing pairing_crs"},
        {"stage": "Raster mask", "filter": "mask mode", "value": "crop=True, filled=False, all_touched=True"},
        {"stage": "Raster value filter", "filter": "nodata removal", "value": "drop raster nodata values"},
        {"stage": "Raster value filter", "filter": "NaN removal", "value": "drop NaN values"},
        {"stage": "Raster summary", "filter": "fire frequency statistic", "value": "use maximum pixel value inside each buffer"},
    ]
)

display(filter_table)


In [ ]:
pairs = gpd.read_file(pairs_path)
ff_ds = rasterio.open(ff_path)

pairs["pre_geom"] = gpd.GeoSeries(pairs["geometry_1_wkt"].apply(wkt.loads), crs="EPSG:4326")
pairs["post_geom"] = gpd.GeoSeries(pairs["geometry_2_wkt"].apply(wkt.loads), crs="EPSG:4326")

print("Pairs CRS:", pairs.crs)
print("Fire frequency raster CRS:", ff_ds.crs)
pairs.head()


In [ ]:
pairs[pre_col_name] = None
pairs[post_col_name] = None

for pcrs, idxs in pairs.groupby("pairing_crs").groups.items():
    if pd.isna(pcrs):
        print(f"Skipping rows with missing pairing_crs: {len(idxs)}")
        continue

    pre_ll_group = gpd.GeoSeries(pairs.loc[idxs, "pre_geom"], crs="EPSG:4326")
    post_ll_group = gpd.GeoSeries(pairs.loc[idxs, "post_geom"], crs="EPSG:4326")

    pre_buff_ll = pre_ll_group.to_crs(pcrs).buffer(buffer_size).to_crs("EPSG:4326")
    post_buff_ll = post_ll_group.to_crs(pcrs).buffer(buffer_size).to_crs("EPSG:4326")

    pairs.loc[idxs, pre_col_name] = pre_buff_ll.values
    pairs.loc[idxs, post_col_name] = post_buff_ll.values

print(f"Created columns: {pre_col_name}, {post_col_name}")


In [ ]:
def extract_info_from_polygon(geom, ds):
    try:
        out_img, _ = mask(ds, [mapping(geom)], crop=True, filled=False, all_touched=True)
    except Exception:
        return None

    data = out_img[0].flatten()
    if ds.nodata is not None:
        data = data[data != ds.nodata]
    data = data[~np.isnan(data)]
    if len(data) == 0:
        return None
    return int(np.max(data))


def to_wkt_column(df, col):
    df[col] = df[col].apply(lambda x: wkt.loads(x) if isinstance(x, str) else x)
    df[col] = gpd.GeoSeries(df[col]).to_wkt()


In [ ]:
pre_ff_list = []
post_ff_list = []

for _, row in tqdm(pairs.iterrows(), total=len(pairs), desc="Extracting fire frequency"):
    pre_geom = row[pre_col_name]
    post_geom = row[post_col_name]

    pre_ff_list.append(extract_info_from_polygon(pre_geom, ff_ds))
    post_ff_list.append(extract_info_from_polygon(post_geom, ff_ds))

pairs["pre_FF"] = pre_ff_list
pairs["post_FF"] = post_ff_list

pairs[["pre_FF", "post_FF"]].head()


In [ ]:
for col in ["pre_geom", "post_geom"]:
    to_wkt_column(pairs, col)

pairs = pairs.drop(columns=[pre_col_name, post_col_name], errors="ignore")
pairs.to_file(output_path, driver="GPKG")
ff_ds.close()

print("Fire frequency extraction complete")
print("Saved to:", output_path)
